# Train VibroPredict — Enzyme Kinetics Prediction

**Run this notebook on Google Colab Pro** (A100 recommended — ProtT5 needs ~3GB VRAM).

This notebook:
1. Clones the repo and installs dependencies (including transformers)
2. Creates a synthetic demo dataset (since KinHub-27k requires manual download)
3. Trains `VibroPredictHybrid` with MM-Drop regularization
4. Runs ablation study (drop each branch)
5. Saves checkpoint for local inference

**Time estimate: ~30-60 minutes on A100**

## 1. Setup

In [ ]:
!git clone https://github.com/jayhemnani9910/nobel-dataintelligence.git
%cd nobel-dataintelligence

!pip install -q -r requirements.txt

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Verify transformers
import transformers
print(f"Transformers: {transformers.__version__}")

## 2. Create Synthetic Demo Dataset

KinHub-27k requires manual download. We create a realistic synthetic dataset
with 1000 samples to demonstrate the full training pipeline.

Replace this cell with real data loading when you have KinHub-27k.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

np.random.seed(42)

# Common amino acids for random sequence generation
AA = 'ACDEFGHIKLMNPQRSTVWY'

# Sample SMILES for common enzyme substrates
SUBSTRATES = [
    'CC(=O)O',          # Acetic acid
    'OC(=O)CC(O)(CC(O)=O)C(O)=O',  # Citric acid
    'OCC1OC(O)C(O)C(O)C1O',  # Glucose
    'CC(=O)SCC(NC(C)=O)C(O)=O',  # Acetyl-CoA simplified
    'NC(CC(O)=O)C(O)=O',  # Aspartate
    'NC(CCC(O)=O)C(O)=O',  # Glutamate
    'OC(=O)C=CC(O)=O',  # Fumarate
    'CC(O)C(O)=O',  # Lactate
]

N_SAMPLES = 1000
data_rows = []
for i in range(N_SAMPLES):
    seq_len = np.random.randint(100, 500)
    sequence = ''.join(np.random.choice(list(AA), seq_len))
    substrate = SUBSTRATES[i % len(SUBSTRATES)]
    # Synthetic k_cat (log-normal distribution, realistic range)
    log_kcat = np.random.normal(1.5, 1.5)  # log10(k_cat), centered ~30 s⁻¹
    mutation = 'WT' if i < N_SAMPLES // 2 else f"{AA[np.random.randint(20)]}{np.random.randint(1, seq_len)}{AA[np.random.randint(20)]}"
    data_rows.append({
        'uniprot_id': f'P{i:05d}',
        'protein_sequence': sequence,
        'k_cat': 10 ** log_kcat,
        'log_kcat': log_kcat,
        'substrate_smiles': substrate,
        'product_smiles': '',
        'organism': 'Synthetic',
        'mutation': mutation,
    })

df = pd.DataFrame(data_rows)
print(f"Dataset: {len(df)} samples")
print(f"log_kcat range: [{df['log_kcat'].min():.2f}, {df['log_kcat'].max():.2f}]")
print(f"Sequence length: {df['protein_sequence'].str.len().describe().to_dict()}")

# Save
DATA_DIR = Path('./data/vibropredict')
DATA_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR = DATA_DIR / 'splits'
SPLITS_DIR.mkdir(exist_ok=True)

# Split 80/10/10
idx = np.random.permutation(len(df))
n_train = int(0.8 * len(df))
n_val = int(0.1 * len(df))
train_df = df.iloc[idx[:n_train]]
val_df = df.iloc[idx[n_train:n_train + n_val]]
test_df = df.iloc[idx[n_train + n_val:]]

train_df.to_csv(SPLITS_DIR / 'train.csv', index=False)
val_df.to_csv(SPLITS_DIR / 'val.csv', index=False)
test_df.to_csv(SPLITS_DIR / 'test.csv', index=False)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
# Generate synthetic VDOS for each sample
from src.spectral_generation import SpectralGenerator

VDOS_DIR = DATA_DIR / 'vdos'
VDOS_DIR.mkdir(exist_ok=True)

sg = SpectralGenerator(freq_min=0, freq_max=500, n_points=1000)

for _, row in df.iterrows():
    uid = row['uniprot_id']
    vdos_path = VDOS_DIR / f"{uid}_vdos.npy"
    if not vdos_path.exists():
        n_modes = min(len(row['protein_sequence']), 100)
        freqs = np.sort(np.random.uniform(5, 450, n_modes))
        vdos = sg.generate_dos(freqs, broadening=5.0)
        np.save(vdos_path, vdos)

print(f"Generated {len(df)} VDOS files in {VDOS_DIR}")

## 3. Create DataLoaders

In [ ]:
from torch.utils.data import DataLoader
from vibropredict.data.enzyme_kinetics_dataset import EnzymeKineticsDataset

train_dataset = EnzymeKineticsDataset(str(SPLITS_DIR / 'train.csv'), str(VDOS_DIR))
val_dataset = EnzymeKineticsDataset(str(SPLITS_DIR / 'val.csv'), str(VDOS_DIR))
test_dataset = EnzymeKineticsDataset(str(SPLITS_DIR / 'test.csv'), str(VDOS_DIR))

BATCH_SIZE = 8  # Small batch to fit ProtT5 in memory

def collate_fn(batch):
    """Custom collate for VibroPredict — keeps strings as lists."""
    return {
        'sequences': [b['sequence'] for b in batch],
        'vdos': torch.stack([b['vdos'] for b in batch]),
        'substrate_smiles': [b['substrate_smiles'] for b in batch],
        'product_smiles': [b['product_smiles'] for b in batch],
        'log_kcat': torch.stack([b['log_kcat'] for b in batch]),
    }

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

# Check one batch
batch = next(iter(train_loader))
print(f"Batch keys: {list(batch.keys())}")
print(f"VDOS shape: {batch['vdos'].shape}")
print(f"Targets shape: {batch['log_kcat'].shape}")

## 4. Initialize Model

In [ ]:
from vibropredict.models.vibropredict_hybrid import VibroPredictHybrid

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = VibroPredictHybrid(
    fusion_dim=512,
    dropout=0.2,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Device: {device}")

## 5. Train with MM-Drop

In [ ]:
from vibropredict.training.trainer import TrainerWithMMDrop
from vibropredict.training.losses import MutantRankingLoss

EPOCHS = 20
LR = 1e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
loss_fn = MutantRankingLoss(lambda_rank=0.1)

CHECKPOINT_DIR = './checkpoints/vibropredict'

trainer = TrainerWithMMDrop(
    model=model,
    optimizer=optimizer,
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
)

best_loss = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    epochs=EPOCHS,
    p_drop=0.25,
    patience=8,
)
print(f"\nBest validation loss: {best_loss:.4f}")

## 6. Evaluate + Ablation

In [ ]:
from vibropredict.training.metrics import compute_all_metrics

# Full evaluation
val_results = trainer.validate(test_loader, loss_fn)
metrics = compute_all_metrics(val_results['predictions'], val_results['targets'])

print("Test Set Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Ablation: test with spectral branch dropped
import matplotlib.pyplot as plt

ablation_results = {}

# Full model
model.eval()
preds_full, targets_all = [], []
with torch.no_grad():
    for batch in test_loader:
        vdos = batch['vdos'].to(device)
        targets = batch['log_kcat']
        logkcat, gates = model(
            batch['sequences'], vdos, batch['substrate_smiles'],
            drop_spectral=False,
        )
        preds_full.append(logkcat.cpu().numpy())
        targets_all.append(targets.numpy())

preds_full = np.concatenate(preds_full)
targets_all = np.concatenate(targets_all)
ablation_results['Full'] = compute_all_metrics(preds_full, targets_all)

# No spectral
preds_nospec = []
with torch.no_grad():
    for batch in test_loader:
        vdos = batch['vdos'].to(device)
        logkcat, _ = model(
            batch['sequences'], vdos, batch['substrate_smiles'],
            drop_spectral=True,
        )
        preds_nospec.append(logkcat.cpu().numpy())
preds_nospec = np.concatenate(preds_nospec)
ablation_results['No Spectral'] = compute_all_metrics(preds_nospec, targets_all)

print("\nAblation Results:")
for variant, m in ablation_results.items():
    print(f"  {variant}: R²={m['r_squared']:.4f}, RMSE={m['rmse']:.4f}, Spearman={m['spearman']:.4f}")

## 7. Visualize

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Predictions vs targets
axes[0].scatter(targets_all, preds_full, alpha=0.3, s=10, c='#22d3ee')
lims = [min(targets_all.min(), preds_full.min()), max(targets_all.max(), preds_full.max())]
axes[0].plot(lims, lims, 'r--', alpha=0.5)
axes[0].set_xlabel('Actual log₁₀(k_cat)'); axes[0].set_ylabel('Predicted')
axes[0].set_title(f"R² = {ablation_results['Full']['r_squared']:.3f}")

# Ablation bar chart
variants = list(ablation_results.keys())
r2_vals = [ablation_results[v]['r_squared'] for v in variants]
colors = ['#22d3ee', '#f472b6']
axes[1].bar(variants, r2_vals, color=colors[:len(variants)])
axes[1].set_ylabel('R²'); axes[1].set_title('Ablation Study')
axes[1].set_ylim(0, max(r2_vals) * 1.2 if max(r2_vals) > 0 else 1)

# Error distribution
errors = preds_full - targets_all
axes[2].hist(errors, bins=40, color='#a78bfa', alpha=0.8)
axes[2].set_xlabel('Error'); axes[2].set_ylabel('Count')
axes[2].set_title(f"MAE = {np.mean(np.abs(errors)):.3f}")

plt.tight_layout(); plt.show()

## 8. Save Checkpoint

In [ ]:
final_path = Path(CHECKPOINT_DIR) / 'vibropredict_best.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_loss': best_loss,
    'metrics': ablation_results,
    'config': {
        'fusion_dim': 512,
        'dropout': 0.2,
        'spec_dim': 128,
        'seq_dim': 1024,
        'chem_dim': 1024,
    },
}, final_path)
print(f"Checkpoint saved: {final_path} ({final_path.stat().st_size / 1024:.0f} KB)")

## 9. Download Checkpoint

In [ ]:
try:
    from google.colab import files
    files.download(str(final_path))
    print("Download started!")
except ImportError:
    print(f"Not on Colab. Checkpoint at: {final_path.resolve()}")

print("\n" + "="*60)
print("DONE! To use locally:")
print("  python -m src.cli predict-kcat \\")
print("    --sequence 'MKTIIALSYIF...' \\")
print("    --smiles 'CC(=O)O' \\")
print(f"    --checkpoint {final_path}")
print("="*60)